In [1]:
from pyspark.sql import SparkSession
from dotenv import load_dotenv
import os
load_dotenv()
print(os.getenv("SPARK_LOCAL_IP"))

127.0.0.1


In [2]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-4")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e252b925-9bf7-49d2-b36f-9fb8b8bd1cca;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 134ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

**Task 1**

Read customers.csv from S3 with an explicit schema. 

Write it to S3 as a CSV file using overwrite mode with a header.

Read it back and verify the row count matches.

In [3]:
from pyspark.sql.types import *

schema=StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("segment", StringType(), True)
])
customers=spark.read.\
    option("header", "true").\
    schema(schema).\
    csv("s3a://pyspark-30-days-rahul-2026/data/customers.csv")
customers.write.\
    mode("overwrite").\
    option("header", "true").\
    csv("s3a://pyspark-30-days-rahul-2026/data/customers_output")
customers_output=spark.read.\
    option("header", "true").\
    schema(schema).\
    csv("s3a://pyspark-30-days-rahul-2026/data/customers_output")
print(customers_output.count()==customers.count())

26/07/27 10:55:55 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/07/27 10:56:04 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/07/27 10:56:09 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


True


**Task 2**

Read orders.csv from S3 with an explicit schema.
 
Write it as Parquet to output/orders_parquet/. 

Read the Parquet back — notice you don't need a schema or header option. Print the schema of the Parquet file.

In [11]:
from pyspark.sql.types import *
schema=StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("region", StringType(), True)
])
orders=spark.read.\
    option("header", "true").\
    schema(schema).\
    csv("s3a://pyspark-30-days-rahul-2026/data/orders.csv")
orders.write.\
    mode("overwrite").\
    option("header", "true").\
    parquet("s3a://pyspark-30-days-rahul-2026/data/orders_output")
orders_output=spark.read.\
    parquet("s3a://pyspark-30-days-rahul-2026/data/orders_output")
orders_output.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- region: string (nullable = true)



**Task 3**


Write orders.csv to S3 as Parquet partitioned by region.

Read back only the East region partition. 

How many rows does the East partition have?

In [13]:
orders.write.\
    mode("overwrite").\
    option("header", "true").\
    parquet("s3a://pyspark-30-days-rahul-2026/data/orders_output_partitioned", partitionBy="region")

orders_east_region=spark.read.\
    parquet("s3a://pyspark-30-days-rahul-2026/data/orders_output_partitioned/region=East")
orders_east_region.count()

27

**Task 4**

Try writing to the same path twice without specifying a mode.

What error do you get? Then fix it by adding the correct write mode.

In [14]:
orders.write.\
    option("header", "true").\
    parquet("s3a://pyspark-30-days-rahul-2026/data/orders_output_partitioned", partitionBy="region")

AnalysisException: [PATH_ALREADY_EXISTS] Path s3a://pyspark-30-days-rahul-2026/data/orders_output_partitioned already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

In [18]:
orders.write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet("s3a://pyspark-30-days-rahul-2026/data/orders_output_partitioned",partitionBy="region")